# Intervention example
In this notebook, let's think about how we might consider and cost an intervention against an infectious disease. We'll consider one illustrative example, being a new vaccine that has recently been invented to combat disease X.

We'll build the model with a simlar approach to what you've already seen, but add a vaccinated compartment that is partially protected against infection. (The vaccine is "leaky" in the sense that the rate at which the vaccinated are infected is reduced, but the vaccinated can still be infected.) This will give us some estimate of the effect of the vaccine in reducing the key modelled quantity we are trying to influence, in this case hospitalisations, and use this to quantify the cost-effectiveness.

![](../images/sir_vacc.svg)

In [ ]:
!pip install git+https://github.com/monash-emu/summer3wip.git@ab4f260aa23d01d1ee50347ea8990c54783ec074

In [ ]:
import numpy as np
from plotly import graph_objects as go

import pandas as pd

pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import (
    Stratification,
    CompartmentMap,
    ManagedArray,
    CategoryData,
    TransitionFlow,
    CompartmentalEpiModel,
)

In [ ]:
population = 1e6
seed = 1.0
start_time = 0
end_time = 50
model_comps = [
    "susceptible",
    "infectious",
    "recovered",
]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))


def infection_process(
    compartment_values: ManagedArray,
    contact_rate: float,
    infectious_compartments: tuple,
):
    infectious_population = compartment_values.query(infectious_compartments).sum()
    total_infectious = compartment_values.sum()
    infectious_prevalence = infectious_population / total_infectious
    return contact_rate * infectious_prevalence


recovery = TransitionFlow(
    "recovery",
    disease_state["infectious"],
    disease_state["recovered"],
    Parameter("recovery_rate", 0.0),
)
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(
    CategoryData(disease_state.categories(), np.array(start_pop))
)

### Stratification
Again, there a few ways we could approach this,
but here we'll stratify just the infectious compartment by hospitalisation status
and just the susceptible compartment according to vaccination status.

In [ ]:
hosp_strata = ["hosp", "non_hosp"]
hospitalise_strat = Stratification("hospitalisation", hosp_strata)
humans.stratify(hospitalise_strat, (disease_state, ["infectious"]))

vacc_strata = ["vacc", "unvacc"]
vacc_strat = Stratification("vaccination", vacc_strata)
humans.stratify(vacc_strat, (disease_state, ["susceptible"]))

### Infection process
Now we'll loop over both vaccination status and hospitalisation status
to adjust the process of moving from each of the susceptible categories
to each of the infectious categories - i.e. four transitions.
The rate of infection for the vaccinated population be adjusted
according to the effectiveness of vaccination on that population,
while these rates of infection will be split according 
to the proportion hospitalised.

In [ ]:
hosp_fraction = Parameter("hosp_fraction", 0.0)
for vacc_status in ["vacc", "unvacc"]:
    for hosp_strat in hosp_strata:
        adjusted_contact_rate = Parameter("effective_contact_rate", 0.0)
        # For the vaccinated group, reduce the force of infection by the vaccination effect
        if vacc_status == "vacc":
            adjusted_contact_rate = adjusted_contact_rate * (
                1.0 - Parameter("vacc_effect", 0.0)
            )

        hosp_prop = hosp_fraction if hosp_strat == "hosp" else 1.0 - hosp_fraction
        split_infection_rate = adjusted_contact_rate * hosp_prop
        force_of_infection = defer(infection_process)(
            CompartmentValues, split_infection_rate, disease_state["infectious"]
        )
        infection = TransitionFlow(
            f"infection_{vacc_status}_{hosp_strat}",
            (disease_state["susceptible"], vacc_strat[vacc_status]),
            (disease_state["infectious"], hospitalise_strat[hosp_strat]),
            force_of_infection,
        )
        epi_model.add_flow(infection)

### Intervention comparison
Next, we need to run the model with and without vaccination to compare the two scenarios.
We'll run it without vaccination by setting the vaccination effect to zero,
and with by setting the vaccination effect to some other effectiveness value.
We can also adjust the coverage through the `hosp_fraction` parameter.

In [ ]:
vacc_effects = [0.0, 0.5]
outputs = pd.DataFrame(columns=vacc_effects)
base_parameters = {
    "effective_contact_rate": 1.2,
    "recovery_rate": 0.2,
    "hosp_fraction": 0.5,
}

for effect in vacc_effects:
    results = epi_model.run(base_parameters | {"vacc_effect": effect})
    outputs[effect] = results["compartments"].to_pandas_df()["infectious_hosp"]

We'll plot the outputs over time for each scenario.

In [ ]:
fig = go.Figure()
legend_format = {"title": "vaccination effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(
        go.Scatter(
            x=outputs.index,
            y=outputs[c],
            mode="lines",
            name=round(c, 1),
            line={"color": colour},
        )
    )
fig.update_layout(
    title="effect of vaccination on the epidemic",
    legend=legend_format,
    xaxis=xaxis_format,
    yaxis={"title": "infection prevalence"},
)

## Cost effectiveness calculations
Now we'll calculate the total costs of the program (in some unspecified unit of currency)
based on a total per capita cost of vaccinating each person.

In [ ]:
n_vaccinated = population * vacc_effects[1]
cost_per_vacc = 50.0
program_cost = n_vaccinated * cost_per_vacc
print(f"Program cost is {round(program_cost / 1e6, 1)} million.")

In [ ]:
cum_hosp_states = outputs.cumsum()
cum_averted_hosps = cum_hosp_states[0.0].iloc[-1] - cum_hosp_states[0.5].iloc[-1]
print(
    f"{round(cum_averted_hosps)} hospitalisations were averted during the simulation period."
)
cum_hosp_states.plot(labels={"index": "time", "value": "cumulative admissions"})

Divide the program cost by the averted hospitalisations to calculate the cost per hospitalisation averted - 
an incremental cost-effectiveness ratio.

In [ ]:
cost_per_hosp_averted = program_cost / cum_averted_hosps
print(f"Cost per hospitalisation averted is {round(cost_per_hosp_averted)}")